In [1]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

df_raw = pd.read_csv("../data/raw/development.csv")


In [4]:
df_raw.columns

Index(['Id', 'source', 'title', 'article', 'page_rank', 'timestamp', 'label'], dtype='object')

In [2]:
def extract_html_tag_features(html):
	if pd.isna(html):
		return pd.Series({
			"n_p": 0,
			"n_h": 0,
			"n_lists": 0,
			"n_blockquote": 0,
			"n_img": 0
		})

	soup = BeautifulSoup(html, "html.parser")

	return pd.Series({
		"n_p": len(soup.find_all("p")),
		"n_h": len(soup.find_all(["h1", "h2", "h3"])),
		"n_lists": len(soup.find_all(["ul", "ol", "li"])),
		"n_blockquote": len(soup.find_all("blockquote")),
		"n_img": len(soup.find_all("img"))
	})

In [5]:
html_features = df_raw["article"].apply(extract_html_tag_features)
df_raw = pd.concat([df_raw[["Id", "label"]], html_features], axis=1)


C:\Users\msist\AppData\Local\Temp\ipykernel_41484\3162279450.py:11: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(html, "html.parser")


In [6]:
for col in ["n_p", "n_h", "n_lists", "n_blockquote", "n_img"]:
	df_raw[f"log_{col}"] = np.log1p(df_raw[col])


In [7]:
df_raw.groupby("label")[
	["log_n_p", "log_n_h", "log_n_lists", "log_n_blockquote", "log_n_img"]
].describe()


log_n_p                                                    log_n_h  \
         count      mean       std  min  25%  50%  75%       max    count   
label                                                                       
0      23542.0  0.122586  0.269621  0.0  0.0  0.0  0.0  2.397895  23542.0   
1      10588.0  0.058697  0.194398  0.0  0.0  0.0  0.0  1.098612  10588.0   
2      11161.0  0.207429  0.490174  0.0  0.0  0.0  0.0  4.043051  11161.0   
3       9977.0  0.081489  0.228514  0.0  0.0  0.0  0.0  2.079442   9977.0   
4       8574.0  0.073729  0.213715  0.0  0.0  0.0  0.0  0.693147   8574.0   
5      13053.0  0.029366  0.139621  0.0  0.0  0.0  0.0  0.693147  13053.0   
6       3102.0  0.033518  0.148716  0.0  0.0  0.0  0.0  0.693147   3102.0   

            ... log_n_blockquote      log_n_img                                \
      mean  ...              75%  max     count      mean       std  min  25%   
label       ...                                                                 
0      0.0  ...              0.0  0.0   23542.0  0.139377  0.317212  0.0  0.0   
1      0.0  ...              0.0  0.0   10588.0  0.101608  0.303911  0.0  0.0   
2      0.0  ...              0.0  0.0   11161.0  0.246464  0.562442  0.0  0.0   
3      0.0  ...              0.0  0.0    9977.0  0.103792  0.317590  0.0  0.0   
4      0.0  ...              0.0  0.0    8574.0  0.084995  0.235308  0.0  0.0   
5      0.0  ...              0.0  0.0   13053.0  0.103863  0.357534  0.0  0.0   
6      0.0  ...              0.0  0.0    3102.0  0.044585  0.200877  0.0  0.0   

                           
       50%  75%       max  
label                      
0      0.0  0.0  2.833213  
1      0.0  0.0  2.079442  
2      0.0  0.0  2.197225  
3      0.0  0.0  3.610918  
4      0.0  0.0  1.791759  
5      0.0  0.0  2.079442  
6      0.0  0.0  1.791759  

[7 rows x 40 columns]

In [8]:
from sklearn.feature_selection import mutual_info_classif

X_html = df_raw[
	["log_n_p", "log_n_h", "log_n_lists", "log_n_blockquote", "log_n_img"]
].fillna(0)

y = df_raw["label"]

mi_html = mutual_info_classif(X_html, y, random_state=42)

pd.Series(mi_html, index=X_html.columns).sort_values(ascending=False)


log_n_img           0.041476
log_n_p             0.025437
log_n_blockquote    0.003240
log_n_lists         0.001550
log_n_h             0.000000
dtype: float64

In [10]:
dataset_token = pd.read_csv("../data/processed/v2/development_token_nohref_in_text.csv")

In [11]:
dataset_html = dataset_token.merge(
	df_raw.drop(columns=["label"]),
	on="Id",
	how="left"
)

dataset_html.fillna(0, inplace=True)


In [14]:
cols_to_drop = [
	"log_n_blockquote",
	"log_n_lists",
	"log_n_h"
]

dataset_final = dataset_html.drop(columns=cols_to_drop)



In [15]:
dataset_final.columns



Index(['Id', 'text', 'source', 'title', 'n_tokens', 'title_ratio', 'year',
       'month', 'has_timestamp', 'label', 'log_n_links', 'source_entropy',
       'source_max_prior', 'source_support', 'n_p', 'n_h', 'n_lists',
       'n_blockquote', 'n_img', 'log_n_p', 'log_n_img'],
      dtype='object')

In [21]:
cols_to_drop = [
	"n_h",
	"n_lists",
	"n_blockquote"
]

dataset_final = dataset_html.drop(columns=cols_to_drop)

In [22]:
dataset_final = dataset_final.drop(columns=["n_p", "n_img"])


In [23]:
X = dataset_final.drop(columns=["label", "Id"])
y = dataset_final["label"]


In [24]:
dataset_final.columns

Index(['Id', 'text', 'source', 'title', 'n_tokens', 'title_ratio', 'year',
       'month', 'has_timestamp', 'label', 'log_n_links', 'source_entropy',
       'source_max_prior', 'source_support', 'log_n_p', 'log_n_h',
       'log_n_lists', 'log_n_blockquote', 'log_n_img'],
      dtype='object')

In [25]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

text_col = "text"

numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns

preprocess = ColumnTransformer(
	transformers=[
		("text", TfidfVectorizer(
			max_features=50000,
			ngram_range=(1, 2),
			min_df=5
		), text_col),
		("num", StandardScaler(), numeric_cols)
	]
)

model = LogisticRegression(
	max_iter=1000,
	n_jobs=-1
)

pipeline = Pipeline([
	("prep", preprocess),
	("clf", model)
])



In [26]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
	n_splits=5,
	shuffle=True,
	random_state=42
)

scores = cross_val_score(
	pipeline,
	X,
	y,
	cv=cv,
	scoring="f1_macro",
	n_jobs=-1
)

print("Fold scores:", scores)
print("Mean macro-F1:", scores.mean())


Fold scores: [0.64393573 0.64179429 0.64956942 0.64044022 0.64489113]
Mean macro-F1: 0.6441261577595971
